# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading and exploration of the **FAIR^2** dataset using the `mlcroissant` library. We'll load data as defined by its Croissant schema, identify record sets and fields by their `@id`, and perform exploration as well as basic analysis.

### Dataset Source
* [Croissant schema JSON-LD](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded:')
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review the available record sets and enumerate their `@id` and their available fields (and columns), as referenced by their `@id` as required for FAIR data processing.


In [ ]:
# List all available record sets and their fields/columns

record_sets = list(dataset.record_sets)
print(f"Available record sets ({len(record_sets)}):\n")
for record_set in record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', '')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")

    # List fields/@id
    if hasattr(record_set, 'fields'):
        print("  Fields and columns:")
        for field in record_set.fields:
            col_id = getattr(field, 'id', None)
            col_label = getattr(field, 'name', None)
            print(f"    - Field @id: {col_id}, name: {col_label}")
    else:
        print("  (No fields listed on this record set)")
    print('')

## 3. Data Extraction
Load tabular data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s you identified above.

In [ ]:
# Select record sets you want to analyze (add their `@id`s explicitly)
# Here, we demonstrate with all available record sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Extract all records for each record set into a pandas DataFrame
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Record set '{record_set_id}' loaded with shape {df.shape}")
        else:
            print(f"No records extracted for record set '{record_set_id}'")
    except Exception as e:
        print(f"Could not process record set '{record_set_id}': {e}")

# List DataFrame columns for the first record set (change index as needed)
if record_set_ids:
    first_rs = record_set_ids[0]
    if first_rs in dataframes:
        print(f"\nColumns for record set '{first_rs}':\n{dataframes[first_rs].columns.tolist()}")
        display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. Use field `@id`s for referencing columns; adapt the IDs from the Data Extraction or Overview outputs above.

Below is an example EDA on a numeric field. Update `numeric_field_id` and `group_field_id` with the `@id` or name of actual numeric and group fields from your chosen record set.

In [ ]:
# Pick a record set for EDA
record_set_id = record_set_ids[0] if record_set_ids else None
if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]

    # You can list column names by uncommenting below
    # print(df.columns.tolist())

    # Choose the actual field (by `@id` or as appears in DataFrame) for numeric analysis
    # Replace the placeholders below with correct field names for your dataset; e.g., 'cr:MW' for molecular weight
    numeric_field_id = None
    group_field_id = None

    # Try to pick a likely numeric column
    for col in df.columns:
        if 'MW' in col or 'Abundance' in col or 'Score' in col:
            numeric_field_id = col
            break

    # Try to pick a likely grouping column
    for col in df.columns:
        if 'Description' in col or 'Class' in col or 'Group' in col:
            group_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        # Filter to values above the mean (if numeric)
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())
            # Normalize
            filtered_df[numeric_field_id + '_normalized'] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

            # Grouping
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
        else:
            print(f"Field '{numeric_field_id}' does not appear to be numeric.")
    else:
        print("No suitable numeric field found in this record set.")
else:
    print("No available record sets with data for EDA.")

## 5. Visualization
Plot data distributions or relationships between numeric and group fields, using the column names identified in EDA. Requires matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # Violin or boxplot by group if grouping field available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.xticks(rotation=90)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()


## 6. Conclusion
In this notebook, we've:
- Loaded FAIR^2 protein mass spectrometry dataset metadata and records using the Croissant schema and `mlcroissant`.
- Explored record sets and fields using their unique `@id` references.
- Loaded tabular data into DataFrames and performed basic filtering and normalization on numeric fields.
- Visualized data distributions and means grouped by annotation or class field.

This template can be extended for deeper biological insight, protein abundance exploration, or custom FAIR-compliant annotation workflows.